# Experiment 4ciii: Hyperparameter Sweep

Systematic sweep of all hyperparameters for the 4c GAT model (LSTM-GATv2 with rolling Pearson mask).

**Sections:**
1. Threshold sweep (graph density)
2. Regularization (dropout, L2, recurrent dropout, feature dropout)
3. GAT capacity (units, heads)
4. LSTM capacity (hidden size)
5. Architectural ablations (residual, scale_scores)
6. Learning rate sensitivity
7. Comparison & analysis

Each run saves to Drive immediately and frees memory. Survives kernel restarts.

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from empyrical import sharpe_ratio, sortino_ratio, max_drawdown, annual_return, annual_volatility, calmar_ratio
import random, tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
SEED = 40
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023
VOL_TARGET = 0.15
TRAIN_STRIDE = 20

# Rolling Pearson Configuration
CORRELATION_LOOKBACK = 20
CORRELATION_THRESHOLD = 0.4
USE_EQUITY_RETURNS = True

# Model Configuration (4c baseline)
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GATv2 Hyperparameters (4c baseline)
HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 16
ATTN_HEADS = 4
LSTM_DROPOUT = 0.4
ATTN_DROPOUT = 0.1
LEARNING_RATE = 0.0008
MAX_GRADIENT_NORM = 1.0
BATCH_SIZE = 57
SCALE_SCORES = True
USE_RESIDUAL = False

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
else:
    RESULTS_BASE = "FINAL_RESULTS"

print(f"Experiment: 4ciii Hyperparameter Sweep")
print(f"Results base: {RESULTS_BASE}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col="captured_returns"):
    num_tickers = df["identifier"].nunique()
    daily_ret = df.groupby("time")[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()

def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    current_vol = daily_returns.std() * np.sqrt(252)
    return daily_returns * (target_vol / current_vol) if current_vol > 0 else daily_returns

def calc_metrics(daily_returns, name="Strategy"):
    return {"Strategy": name, "E[Ret.]": annual_return(daily_returns),
        "Vol.": annual_volatility(daily_returns), "Sharpe": sharpe_ratio(daily_returns),
        "Sortino": sortino_ratio(daily_returns), "Max DD": -max_drawdown(daily_returns),
        "Calmar": calmar_ratio(daily_returns), "Hit Rate": (daily_returns > 0).mean(),
        "Avg P/L": daily_returns[daily_returns > 0].mean() / abs(daily_returns[daily_returns < 0].mean()) if (daily_returns < 0).any() else np.nan}

def calc_metrics_vol_normalized(daily_returns, name, target_vol=0.15):
    scaled = calc_vol_scaled_returns(daily_returns, target_vol)
    return calc_metrics(scaled, name + " (Vol-Norm)"), scaled

def calc_yearly_sharpes(daily_returns):
    return {y: sharpe_ratio(daily_returns[daily_returns.index.year == y]) for y in sorted(daily_returns.index.year.unique())}

## 4. Data Loading

In [ ]:
features_path = "data/straddle_features/features.csv"
if 'google.colab' in str(get_ipython()):
    features_path = "/content/drive/MyDrive/features.csv"

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
from gml.graph_model_inputs import RollingGraphModelFeatures

def load_data(threshold=CORRELATION_THRESHOLD):
    """Load rolling graph data with given threshold. Returns datasets dict."""
    rolling_features = RollingGraphModelFeatures(
        df=df,
        total_time_steps=TOTAL_TIME_STEPS,
        correlation_lookback=CORRELATION_LOOKBACK,
        correlation_threshold=threshold,
        returns_column="daily_returns",
        use_equity_returns=USE_EQUITY_RETURNS,
        start_boundary=TRAIN_START,
        test_boundary=TEST_START,
        test_end=TEST_END,
        train_valid_ratio=TRAIN_VALID_RATIO,
        split_tickers_individually=True,
        time_features=False,
    )
    return rolling_features.make_rolling_graph_dataset(sliding_window=True)

# Load default data (threshold=0.4)
datasets = load_data(CORRELATION_THRESHOLD)
train_data = datasets["train"]
valid_data = datasets["valid"]
test_data = datasets["test"]

num_tickers = train_data["inputs"].shape[1]
time_steps = train_data["inputs"].shape[2]
input_size = train_data["inputs"].shape[3]

print(f"Training: {train_data['inputs'].shape}")
print(f"Test: {test_data['inputs'].shape}")
print(f"num_tickers={num_tickers}, time_steps={time_steps}, input_size={input_size}")

## 5. Sweep Infrastructure

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_rolling, extract_attention_weights_v2
from gml.experiment_utils import save_experiment_results, load_experiment_results, compute_graph_stats

all_results = {}

def run_sweep(config_name, category, threshold_override=None, **model_kwargs):
    """Train one sweep variant, save to Drive, store lightweight results, free memory.
    
    Args:
        config_name: Name for this run (e.g., 'th_0.3', 'lstm_6')
        category: Sweep category for directory structure (e.g., '4ciii_threshold_sweep')
        threshold_override: If set, regenerate data with this threshold
        **model_kwargs: Override any build_lstm_gat_rolling parameter
    """
    global train_data, valid_data, test_data
    
    # Check if already saved on Drive
    existing = load_experiment_results(category, SEED, config_name=config_name, base_dir=RESULTS_BASE)
    if existing is not None and existing.get("daily_returns") is not None:
        print(f"  {config_name}: already saved, loading from Drive")
        daily_returns = existing["daily_returns"]
        metrics_raw = calc_metrics(daily_returns, config_name)
        all_results[f"{category}/{config_name}"] = {
            "daily_returns": daily_returns,
            "metrics": metrics_raw,
            "category": category,
            "config_name": config_name,
        }
        print(f"  Sharpe: {metrics_raw['Sharpe']:.4f} (loaded)")
        return
    
    print("=" * 60)
    print(f"RUN: {category} / {config_name}")
    print(f"  Overrides: {model_kwargs}")
    print("=" * 60)
    
    # Regenerate data if threshold changed
    if threshold_override is not None and threshold_override != CORRELATION_THRESHOLD:
        print(f"  Regenerating data with threshold={threshold_override}...")
        sweep_datasets = load_data(threshold_override)
        s_train = sweep_datasets["train"]
        s_valid = sweep_datasets["valid"]
        s_test = sweep_datasets["test"]
    else:
        s_train, s_valid, s_test = train_data, valid_data, test_data
    
    X_train = [s_train["inputs"], s_train["adjacency"]]
    y_train = s_train["outputs"]
    w_train = s_train["active_entries"]
    X_valid = [s_valid["inputs"], s_valid["adjacency"]]
    y_valid = s_valid["outputs"]
    w_valid = s_valid["active_entries"]
    X_test = [s_test["inputs"], s_test["adjacency"]]
    
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    
    # Merge defaults with overrides
    build_kwargs = dict(
        num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
        hidden_layer_size=HIDDEN_LAYER_SIZE, gat_units=GAT_UNITS, attn_heads=ATTN_HEADS,
        lstm_dropout=LSTM_DROPOUT, attn_dropout=ATTN_DROPOUT,
        learning_rate=LEARNING_RATE, max_gradient_norm=MAX_GRADIENT_NORM,
        scale_scores=SCALE_SCORES, use_residual=USE_RESIDUAL,
    )
    build_kwargs.update(model_kwargs)
    
    model = build_lstm_gat_rolling(**build_kwargs)
    print(f"  Parameters: {model.count_params():,}")
    
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True, verbose=1,
    )
    
    history = model.fit(
        X_train, y_train, sample_weight=w_train,
        validation_data=(X_valid, y_valid, w_valid),
        epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[early_stopping], verbose=1,
    )
    
    predictions = model.predict(X_test)
    
    # Build results
    positions = predictions[:, :, -1, 0].reshape(-1)
    returns = s_test["outputs"][:, :, -1, 0].reshape(-1)
    captured_returns = positions * returns
    dates = s_test["date"][:, :, -1, 0].reshape(-1)
    identifiers = s_test["identifier"][:, :, -1, 0].reshape(-1)
    
    results_df = pd.DataFrame({
        "time": dates, "identifier": identifiers,
        "position": positions, "returns": returns, "captured_returns": captured_returns,
    })
    results_df["time"] = pd.to_datetime(results_df["time"])
    results_df = results_df[results_df["identifier"] != "0"]
    
    daily_returns = calc_daily_returns(results_df)
    metrics_raw = calc_metrics(daily_returns, config_name)
    metrics_norm, _ = calc_metrics_vol_normalized(daily_returns, config_name, VOL_TARGET)
    yearly_sharpes = calc_yearly_sharpes(daily_returns)
    test_dates_arr = pd.to_datetime(s_test["date"][:, 0, -1, 0])
    
    # Extract attention
    try:
        attn_weights = extract_attention_weights_v2(model, X_test, gat_layer_name="dynamic_masked_gat")
    except Exception as e:
        print(f"  Warning: attention extraction failed: {e}")
        attn_weights = None
    
    # Save to Drive
    hyperparams = {**build_kwargs, "threshold_override": threshold_override}
    save_experiment_results(
        experiment_name=category, seed=SEED, config_name=config_name,
        predictions=predictions, results_df=results_df,
        daily_returns=daily_returns, metrics_raw=metrics_raw,
        metrics_norm=metrics_norm, yearly_sharpes=yearly_sharpes,
        training_history=history.history, hyperparams=hyperparams,
        test_dates=test_dates_arr.values,
        attention_weights=attn_weights,
        adjacency=s_test["adjacency"],
        model=model,
        base_dir=RESULTS_BASE,
    )
    print(f"  Saved: {category}/{config_name}")
    
    # Store lightweight results
    all_results[f"{category}/{config_name}"] = {
        "daily_returns": daily_returns,
        "metrics": metrics_raw,
        "history_loss": history.history["loss"],
        "history_val_loss": history.history["val_loss"],
        "category": category,
        "config_name": config_name,
    }
    
    # Free memory
    del model, predictions, attn_weights, results_df
    tf.keras.backend.clear_session()
    import gc; gc.collect()
    
    print(f"  Sharpe: {metrics_raw['Sharpe']:.4f}")
    print(f"  Train-Val gap: {abs(history.history['loss'][-1]) - abs(history.history['val_loss'][-1]):.4f}")
    print()

## 6. Threshold Sweep (HIGH PRIORITY)

Currently 4c uses threshold=0.4 while GCN uses 0.5. Sweep to find optimal and enable fair comparison.

In [ ]:
for th in [0.3, 0.4, 0.5, 0.6]:
    run_sweep(f"th_{th}", "4ciii_threshold_sweep", threshold_override=th)

## 7. Regularization Sweeps

In [ ]:
# Attention dropout
for ad in [0.0, 0.1, 0.3, 0.5]:
    run_sweep(f"attn_drop_{ad}", "4ciii_regularization", attn_dropout=ad)

In [ ]:
# LSTM dropout
for ld in [0.2, 0.4, 0.6]:
    run_sweep(f"lstm_drop_{ld}", "4ciii_regularization", lstm_dropout=ld)

In [ ]:
# Recurrent dropout (NEW param)
for rd in [0.1, 0.2, 0.3]:
    run_sweep(f"rec_drop_{rd}", "4ciii_regularization", recurrent_dropout=rd)

In [ ]:
# L2 weight regularization (NEW param)
for l2 in [1e-4, 1e-3, 1e-2]:
    run_sweep(f"l2_{l2}", "4ciii_regularization", l2_lambda=l2)

In [ ]:
# Feature dropout (NEW param)
for fd in [0.1, 0.2]:
    run_sweep(f"feat_drop_{fd}", "4ciii_regularization", feature_dropout=fd)

In [ ]:
# Combined regularization
run_sweep("reg_combined", "4ciii_regularization",
          attn_dropout=0.3, recurrent_dropout=0.2, l2_lambda=1e-3)

## 8. GAT Capacity Sweeps

In [ ]:
# GAT units
for u in [8, 16, 32]:
    run_sweep(f"gat_units_{u}", "4ciii_gat_capacity", gat_units=u)

In [ ]:
# Attention heads
for h in [1, 2, 4, 8]:
    run_sweep(f"heads_{h}", "4ciii_gat_capacity", attn_heads=h)

## 9. LSTM Capacity Sweeps

In [ ]:
# LSTM hidden size
for hs in [6, 10, 20, 32]:
    run_sweep(f"lstm_{hs}", "4ciii_lstm_capacity", hidden_layer_size=hs)

## 10. Architectural Ablations

In [ ]:
run_sweep("with_residual", "4ciii_architecture", use_residual=True)
run_sweep("no_scale_scores", "4ciii_architecture", scale_scores=False)
run_sweep("residual_no_scale", "4ciii_architecture", use_residual=True, scale_scores=False)

## 11. Learning Rate Sensitivity

In [ ]:
for lr in [0.0003, 0.0008, 0.002]:
    run_sweep(f"lr_{lr}", "4ciii_learning_rate", learning_rate=lr)

---
## 12. Load & Compare

Reload all saved results from Drive (run after kernel restart).

In [ ]:
# Reload all 4ciii results from Drive
CATEGORIES = {
    "4ciii_threshold_sweep": [f"th_{t}" for t in [0.3, 0.4, 0.5, 0.6]],
    "4ciii_regularization": (
        [f"attn_drop_{d}" for d in [0.0, 0.1, 0.3, 0.5]] +
        [f"lstm_drop_{d}" for d in [0.2, 0.4, 0.6]] +
        [f"rec_drop_{d}" for d in [0.1, 0.2, 0.3]] +
        [f"l2_{l}" for l in ["0.0001", "0.001", "0.01"]] +
        [f"feat_drop_{d}" for d in [0.1, 0.2]] +
        ["reg_combined"]
    ),
    "4ciii_gat_capacity": [f"gat_units_{u}" for u in [8, 16, 32]] + [f"heads_{h}" for h in [1, 2, 4, 8]],
    "4ciii_lstm_capacity": [f"lstm_{h}" for h in [6, 10, 20, 32]],
    "4ciii_architecture": ["with_residual", "no_scale_scores", "residual_no_scale"],
    "4ciii_learning_rate": [f"lr_{l}" for l in [0.0003, 0.0008, 0.002]],
}

all_results = {}
for category, configs in CATEGORIES.items():
    for config_name in configs:
        try:
            r = load_experiment_results(category, SEED, config_name=config_name, base_dir=RESULTS_BASE)
            if r is not None and r.get("daily_returns") is not None:
                daily_returns = r["daily_returns"]
                metrics_raw = calc_metrics(daily_returns, config_name)
                hist = r.get("training_history", {})
                all_results[f"{category}/{config_name}"] = {
                    "daily_returns": daily_returns,
                    "metrics": metrics_raw,
                    "history_loss": hist.get("loss", []),
                    "history_val_loss": hist.get("val_loss", []),
                    "category": category,
                    "config_name": config_name,
                }
                print(f"  {category}/{config_name}: Sharpe={metrics_raw['Sharpe']:.4f}")
        except Exception as e:
            pass  # not yet run

print(f"\nLoaded {len(all_results)} runs")

In [ ]:
# Master comparison table
rows = []
for key, r in sorted(all_results.items()):
    m = r["metrics"]
    train_loss = r.get("history_loss", [None])[-1]
    val_loss = r.get("history_val_loss", [None])[-1]
    gap = abs(train_loss) - abs(val_loss) if train_loss and val_loss else None
    rows.append({
        "Category": r["category"].replace("4ciii_", ""),
        "Config": r["config_name"],
        "Sharpe": m["Sharpe"],
        "Sortino": m["Sortino"],
        "Ann. Return": m["E[Ret.]"],
        "Ann. Vol": m["Vol."],
        "Max DD": m["Max DD"],
        "Train-Val Gap": gap,
    })

comp_df = pd.DataFrame(rows)
print("Master Comparison Table (sorted by Sharpe):")
print(comp_df.sort_values("Sharpe", ascending=False).to_string(index=False, float_format="%.4f"))

In [ ]:
# Per-category plots
def plot_category(category_key, x_param, x_values, title):
    """Plot Sharpe vs a swept parameter for one category."""
    sharpes = []
    labels = []
    for config_name in x_values:
        key = f"{category_key}/{config_name}"
        if key in all_results:
            sharpes.append(all_results[key]["metrics"]["Sharpe"])
            labels.append(config_name)
    
    if not sharpes:
        print(f"  No results for {category_key}")
        return
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(range(len(sharpes)), sharpes, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(sharpes))))
    ax.set_xticks(range(len(sharpes)))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Sharpe Ratio")
    ax.set_title(title)
    ax.axhline(y=all_results.get(f"{category_key}/{'th_0.4' if 'threshold' in category_key else labels[len(labels)//2]}", {}).get("metrics", {}).get("Sharpe", 0),
               color="red", ls="--", alpha=0.5, label="baseline")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

# Threshold sweep
plot_category("4ciii_threshold_sweep",
              "threshold", [f"th_{t}" for t in [0.3, 0.4, 0.5, 0.6]],
              "Threshold Sweep: Sharpe vs Graph Density")

# GAT units
plot_category("4ciii_gat_capacity",
              "gat_units", [f"gat_units_{u}" for u in [8, 16, 32]],
              "GAT Units Sweep")

# Attention heads
plot_category("4ciii_gat_capacity",
              "heads", [f"heads_{h}" for h in [1, 2, 4, 8]],
              "Attention Heads Sweep")

# LSTM hidden
plot_category("4ciii_lstm_capacity",
              "hidden", [f"lstm_{h}" for h in [6, 10, 20, 32]],
              "LSTM Hidden Size Sweep")

# Learning rate
plot_category("4ciii_learning_rate",
              "lr", [f"lr_{l}" for l in [0.0003, 0.0008, 0.002]],
              "Learning Rate Sweep")

In [ ]:
# Regularization comparison: bar chart of all reg variants
reg_results = {k: v for k, v in all_results.items() if "regularization" in k}
if reg_results:
    names = [v["config_name"] for v in reg_results.values()]
    sharpes = [v["metrics"]["Sharpe"] for v in reg_results.values()]
    gaps = []
    for v in reg_results.values():
        tl = v.get("history_loss", [None])[-1]
        vl = v.get("history_val_loss", [None])[-1]
        gaps.append(abs(tl) - abs(vl) if tl and vl else 0)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Sharpe
    sort_idx = np.argsort(sharpes)[::-1]
    axes[0].barh([names[i] for i in sort_idx], [sharpes[i] for i in sort_idx],
                 color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(names))))
    axes[0].set_xlabel("Sharpe Ratio")
    axes[0].set_title("Regularization: Sharpe Ratio")
    axes[0].grid(True, alpha=0.3, axis="x")
    
    # Train-Val gap (lower = less overfitting)
    sort_idx2 = np.argsort(gaps)
    axes[1].barh([names[i] for i in sort_idx2], [gaps[i] for i in sort_idx2],
                 color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(names))))
    axes[1].set_xlabel("Train-Val Loss Gap (lower = less overfitting)")
    axes[1].set_title("Regularization: Overfitting")
    axes[1].grid(True, alpha=0.3, axis="x")
    
    plt.suptitle("4ciii: Regularization Sweep", fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# Architectural ablations table
arch_results = {k: v for k, v in all_results.items() if "architecture" in k}
if arch_results:
    print("Architectural Ablations:")
    for key, v in arch_results.items():
        m = v["metrics"]
        print(f"  {v['config_name']:25s} Sharpe={m['Sharpe']:.4f}  Sortino={m['Sortino']:.4f}")
    
    # Compare with baseline
    baseline_key = [k for k in all_results if "th_0.4" in k or ("attn_drop_0.1" in k)]
    if baseline_key:
        bm = all_results[baseline_key[0]]["metrics"]
        print(f"\n  {'baseline (4c)':25s} Sharpe={bm['Sharpe']:.4f}  Sortino={bm['Sortino']:.4f}")